# Processing and Cleaning ClinVar and dbNSFP data

In [1]:
import os
import subprocess
import polars as pl
import pandas as pd
import re

# Set strict error handling
def run_command(cmd):
    result = subprocess.run(cmd, shell=True, check=True, text=True)
    return result

## 1. ClinVar 2025-01

Use ```../1_inputs/download_data.sh``` to download correct version of ClinVar data from https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2025-01.txt.gz

In [7]:
dest_path = "../1_inputs/raw_inputs/clinvar/variant_summary_2025-01.txt.gz"
## Define the new processed and cleaned clinvar file
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"

In [9]:
pl.enable_string_cache()

## Read CSV with Polars (auto-decompresses .gz)
clinvar = pl.read_csv(dest_path, separator='\t', infer_schema_length=1000000)
## Process data with Polars expressions
clinvar = (
    clinvar
    .filter(pl.col("Assembly") == "GRCh38")
    .with_columns(
        pl.col("Name")
        # Split the string on '.', '(' or ')'
        .map_elements(lambda s: re.split(r"[.\(\)]", s), pl.List(pl.String))
        .alias("Name_listed")
    )
)
clinvar = clinvar.with_columns(
    pl.col("Name_listed").list.get(0, null_on_oob=True).alias("NM"),
    pl.col("Name_listed").list.get(1, null_on_oob=True).alias("version"), 
    pl.col("Name_listed").list.get(2, null_on_oob=True).alias("gene"),
    pl.col("Name_listed").list.get(3, null_on_oob=True).alias("c"),
    pl.col("Name_listed").list.get(4, null_on_oob=True).alias("clinvar_nt_change"),
    pl.col("Name_listed").list.get(5, null_on_oob=True).alias("p"),
    pl.col("Name_listed").list.get(6, null_on_oob=True).alias("clinvar_aa_change")
).with_columns(
        SPDI=pl.col("ChromosomeAccession") + ":" + 
            pl.col("PositionVCF").cast(str) + ":" +
            pl.col("ReferenceAlleleVCF") + ":" +
            pl.col("AlternateAlleleVCF"),
        RefSeq_mRNA=pl.col("NM") + "." + pl.col("version"),
        StarStatus=pl.when(pl.col("ReviewStatus").str.contains(r'single|conflicting'))
            .then(1)
            .when(pl.col("ReviewStatus").str.contains('multiple'))
            .then(2)
            .when(pl.col("ReviewStatus").str.contains('expert'))
            .then(3)
            .when(pl.col("ReviewStatus").str.contains('practice'))
            .then(4)
            .otherwise(None)
)

## Create the mapping expression with precise logic for a cleaner ClinVar Annotation
clinvar_mapping = (
    # First check for Conflicting interpretations (highest priority)
    pl.when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("conflicting"))
    .then(pl.lit("Conflicting"))
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("uncertain"))
    .then(pl.lit("VUS"))
    # Check for Pathogenic
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("pathogenic"))
    .then(pl.lit("Pathogenic"))
    # Check for Benign
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("benign"))
    .then(pl.lit("Benign"))
    # Everything else goes to Others
    .otherwise(pl.lit("Others"))
)

## Create the mapping expression with precise logic for a cleaner ClinVar Annotation USING Pillar team's strict Clinical Significance priority
clinvar_mapping_strict_pp = (
    # Pathogenic / Likely pathogenic
    pl.when(pl.col("ClinicalSignificance")=="Pathogenic")
    .then(pl.lit("Pathogenic"))
    .when(pl.col("ClinicalSignificance")=="Pathogenic/Likely pathogenic")
    .then(pl.lit("Likely pathogenic"))
    .when(pl.col("ClinicalSignificance")=="Likely pathogenic")
    .then(pl.lit("Likely pathogenic"))

    # Benign / Likely benign
    .when(pl.col("ClinicalSignificance")=="Benign")
    .then(pl.lit("Benign"))
    .when(pl.col("ClinicalSignificance")=="Benign/Likely benign")
    .then(pl.lit("Likely benign"))
    .when(pl.col("ClinicalSignificance")=="Likely benign")
    .then(pl.lit("Likely benign"))
    
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance")=="Uncertain significance")
    .then(pl.lit("VUS"))
    
    # Check for Conflicting
    .when(pl.col("ClinicalSignificance")=="Conflicting classifications of pathogenicity")
    .then(pl.lit("Conflicting"))

    # Everything else goes to Others
    .otherwise(pl.lit("Others"))
)

# Apply the mapping to create a new column
clinvar = clinvar.with_columns(
    clinvar_clnsig_clean=clinvar_mapping,
    clinvar_clnsig_clean_pp_strict=clinvar_mapping_strict_pp)

# Create a dataframe with unique mappings to check accuracy
# Sort by original value for easier verification
unique_mappings = clinvar.select(
    pl.col("ClinicalSignificance"),
    pl.col("clinvar_clnsig_clean"),
    pl.col("clinvar_clnsig_clean_pp_strict")
).unique().sort("ClinicalSignificance")

# Print each mapping to verify accuracy
print("Original Value -> Mapped Value -> Mapped Value (Pillar Team Strict)")
print("-" * 70)
for row in unique_mappings.rows():
    print(f"{row[0]} -> {row[1]} -> {row[2]} ")

## Make sure we only drop columns that actually exist
columns_to_drop = [col for col in ['NM', 'version', 'gene', 'c', 'p', 'ReferenceAllele', 'AlternateAllele', 'Name_listed'] 
                  if col in clinvar.columns]
if columns_to_drop:
    clinvar = clinvar.drop(columns_to_drop)

# Now cast to categorical
clinvar = clinvar.with_columns(
    pl.col(["Assembly", "ClinicalSignificance", "ReviewStatus", "clinvar_clnsig_clean", "clinvar_clnsig_clean_pp_strict"]).cast(pl.Categorical)
)

Original Value -> Mapped Value -> Mapped Value (Pillar Team Strict)
----------------------------------------------------------------------
- -> Others -> Others 
Affects -> Others -> Others 
Affects; risk factor -> Others -> Others 
Benign -> Benign -> Benign 
Benign/Likely benign -> Benign -> Likely benign 
Benign/Likely benign; association -> Benign -> Others 
Benign/Likely benign; drug response -> Benign -> Others 
Benign/Likely benign; drug response; other -> Benign -> Others 
Benign/Likely benign; other -> Benign -> Others 
Benign/Likely benign; other; risk factor -> Benign -> Others 
Benign/Likely benign; risk factor -> Benign -> Others 
Benign; Affects -> Benign -> Others 
Benign; Affects; association; other -> Benign -> Others 
Benign; association -> Benign -> Others 
Benign; confers sensitivity -> Benign -> Others 
Benign; drug response -> Benign -> Others 
Benign; other -> Benign -> Others 
Benign; protective -> Benign -> Others 
Benign; risk factor -> Benign -> Others 
Confl

In [ ]:
# Write to Parquet with ZSTD compression
# clinvar.write_parquet(processed_clinvar_parq, 
#                      compression="zstd", 
#                      statistics=True)

## 2. dbNSFP v5.0a

Using ```../1_inputs/download_data.sh``` to download data from https://drive.google.com/file/d/1HNoZUswadC1ODavU7AUvm0DZ13jKVFKW/view

We also use dbNSFP for gnomad allele frequency as well, which used gnomadv4.1 for annotations. Documentation at: https://www.dbnsfp.org/releases

In [ ]:
# def convert_to_parquet(chr_num):
#     file_path = f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.gz"
#     df = pl.read_csv(file_path, 
#                      separator='\t',
#                      n_threads=16,
#                      infer_schema_length=1000000,
#                      ignore_errors=True,
#                      truncate_ragged_lines=True,
#                      low_memory=False)
#     df = df.with_columns(
#         pl.col("aaref").replace(aa_single_to_three).alias("aaref_3"),
#         pl.col("aaalt").replace(aa_single_to_three).alias("aaalt_3"),
#     )
#     parquet_path = f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet"
#     df.write_parquet(parquet_path)
#     return parquet_path

# chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_inputs/dbNSFP5.0a') if "chr" in chr_file and chr_file.endswith(".gz")]
# # Use ThreadPoolExecutor for parallel processing
# with concurrent.futures.ThreadPoolExecutor() as executor:
#     futures = [executor.submit(convert_to_parquet, chr_num) for chr_num in chr_num_list]
#     # Collect the results
#     parquet_files = [future.result() for future in concurrent.futures.as_completed(futures)]

# print("Conversion to Parquet completed.")

In [12]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
clinvar = pl.scan_parquet(processed_clinvar_parq)
clinvar_snv = clinvar.filter(pl.col("Type")=='single nucleotide variant') ##.select(pl.col("Type")).unique().collect().to_numpy()
clinvar_snv.collect().shape

(2793330, 45)

### Merge dbNSFP and ClinVar

dbNSFP and ClinVar datasets were merged together using ```VariantionID``` (ClinVar) and ```clinvar_id``` (dbNSFP), which are the unique identifier for each variant.

In [ ]:
# %%time

# with pl.StringCache():
#     chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_inputs/dbNSFP5.0a/') if "chr" in chr_file and chr_file.endswith(".gz")]
#     clin_var_dfs = []
#     for chr_num in chr_num_list:
#         df_chr_par = pl.scan_parquet(f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet")
#         df_chr_par_clinvar = df_chr_par.filter(pl.col("clinvar_id")!='.')
#         clinvar_per_chr = clinvar_snv.filter(pl.col("Chromosome")==chr_num)#.collect()
#         clinvar_per_chr = clinvar_per_chr.with_columns(pl.col("VariationID").cast(pl.String).alias("clinvar_id"))
#         df_chr_par_clinvar = df_chr_par_clinvar.join(clinvar_per_chr, on="clinvar_id", how="left")
#         df_chr_par_clinvar = df_chr_par_clinvar.collect()
#         clin_var_dfs.append(df_chr_par_clinvar)

#     df_clinvar = pl.concat(clin_var_dfs, how="vertical_relaxed")
#     parquet_path = f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot.parquet"
#     df_clinvar.write_parquet(parquet_path)

CPU times: user 45min 44s, sys: 2h 53min 5s, total: 3h 38min 50s
Wall time: 2min 24s


In [15]:
df_clinvar = pl.scan_parquet("../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot.parquet")
# For testing purposes, limit to a small subset:
df_clin_subset = df_clinvar.limit(10)
# Collect the subset into an eager DataFrame for inspection
df_clin_subset = df_clin_subset.collect()

# Automatically detect columns with semicolon-separated patterns.
# We only check columns of Utf8 (string) type.
pattern_cols = [
    col for col in df_clinvar.collect_schema() 
    if df_clin_subset[col].dtype == pl.Utf8 and df_clin_subset[col].str.contains(";").any()
]
print("Columns containing ';':", pattern_cols)

df_clinvar_exploded = df_clinvar.with_columns(
    [pl.col(col).str.split(";").alias(col) for col in pattern_cols]
).explode(pattern_cols)

df_clinvar_exploded.collect().shape

Columns containing ';': ['aapos', 'genename', 'Ensembl_geneid', 'Ensembl_transcriptid', 'Ensembl_proteinid', 'Uniprot_acc', 'Uniprot_entry', 'HGVSc_snpEff', 'HGVSp_snpEff', 'HGVSc_VEP', 'HGVSp_VEP', 'APPRIS', 'GENCODE_basic', 'TSL', 'VEP_canonical', 'MANE', 'cds_strand', 'refcodon', 'codonpos', 'codon_degeneracy', 'SIFT_score', 'SIFT_pred', 'SIFT4G_score', 'SIFT4G_pred', 'Polyphen2_HDIV_score', 'Polyphen2_HDIV_pred', 'Polyphen2_HVAR_score', 'Polyphen2_HVAR_pred', 'MutationTaster_score', 'MutationTaster_pred', 'MutationTaster_model', 'MutationTaster_trees_benign', 'MutationTaster_trees_deleterious', 'MutationAssessor_score', 'MutationAssessor_pred', 'PROVEAN_score', 'PROVEAN_pred', 'VEST4_score', 'MetaRNN_score', 'MetaRNN_pred', 'REVEL_score', 'MutPred_score', 'MutPred_protID', 'MutPred_AAchange', 'MutPred_Top5features', 'MVP_score', 'gMVP_score', 'MPC_score', 'DEOGEN2_score', 'DEOGEN2_pred', 'LIST-S2_score', 'LIST-S2_pred', 'VARITY_R_score', 'VARITY_ER_score', 'VARITY_R_LOO_score', 'VA

(7475238, 427)

### De-duplicate them 

__By prioritizing the uniprot_id with a .pdb structure inferred by AF2__

In [17]:
def deduplicate_by_uniprot_lazy(ldf, preferred_uniprot_acc):
    # Create window specs for our operations
    window_spec = pl.col("clinvar_id")
    
    with pl.StringCache():
        # Add useful columns for filtering in a single pass:
        # 1. Count occurrences of each clinvar_id
        # 2. Check if Uniprot_acc is preferred
        # 3. Check if Uniprot_acc has no dash
        # 4. Create row number within each group to use for selection
        enhanced_ldf = ldf.with_columns([
            pl.col("clinvar_id").count().over(window_spec).alias("group_count"),
            pl.col("Uniprot_acc").is_in(preferred_uniprot_acc).alias("is_preferred"),
            (~pl.col("Uniprot_acc").str.contains("-")).alias("no_dash"),
            pl.col("Uniprot_acc").rank("dense").over(window_spec).alias("row_rank")
        ])
        
        # For non-duplicated entries - keep all
        non_duplicated = enhanced_ldf.filter(pl.col("group_count") == 1)
        
        # For duplicated entries, we'll create a priority column and then use that for selection
        duplicated = enhanced_ldf.filter(pl.col("group_count") > 1)

        # Create a priority column (1=preferred, 2=no dash, 3=other)
        prioritized = duplicated.with_columns([
            pl.when(pl.col("is_preferred"))
            .then(pl.lit(1))
            .when(pl.col("no_dash"))
            .then(pl.lit(2))
            .otherwise(pl.lit(3))
            .alias("priority")
        ])

        # Sort by priority and take the first record for each ID
        deduped = (
            prioritized
            .sort(["clinvar_id", "priority", "row_rank"])
            .group_by("clinvar_id")
            .agg(
                pl.all().exclude(["group_count", "is_preferred", "no_dash", "row_rank", "priority"]).first()
            )
        )

    deduped_df = pl.concat([non_duplicated.drop(["group_count", "is_preferred", "no_dash", "row_rank"]).select(pl.col(deduped.collect_schema().names())),
    deduped], how="vertical_relaxed")
    
    # Combine all results
    return deduped_df

uniprot_ids = [f.split('-')[1] for f in os.listdir("../1_inputs/raw_inputs/alphafold/UP000005640_9606_HUMAN_v4") if f.endswith("pdb.gz")]
## Let's prioritize the variants with PDB structures
df_clinvar_dedup_struct = deduplicate_by_uniprot_lazy(df_clinvar_exploded, uniprot_ids)
df_clinvar_dedup_struct.collect().shape

(1649580, 427)

In [ ]:
# with pl.StringCache():
#     df_clinvar_dedup_struct.collect().write_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_dedup_variants.parquet")